# CRISPR Guide Enumeration

![CRISPR Guide Enumeration](https://proto-bio.github.io/proto-assets/images/tool/crispr_guides/hero.png)

This notebook demonstrates `run_crispr_guides_enumerate`, which scans reference-oriented DNA intervals for every protospacer sitting next to a PAM. It uses deterministic string matching and returns the full candidate pool for ranking with other tools. It does not predict activity, search for off-targets, or select candidates.

In [1]:
from proto_tools.utils.notebook_docs import display_api_reference
from proto_tools.tools.gene_annotation.crispr_guides import (
    CRISPRGuideEnumerationConfig,
    CRISPRGuideEnumerationInput,
    CRISPRGuideRegion,
    run_crispr_guides_enumerate,
)

## API Reference

In [2]:
display_api_reference("crispr-guides-enumerate", "input", "run_crispr_guides_enumerate")

**Input** — `CRISPRGuideEnumerationInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>regions</code> | <code>list[CRISPRGuideRegion]</code> | required | Reference-oriented DNA intervals to scan |

In [3]:
display_api_reference("crispr-guides-enumerate", "config", "run_crispr_guides_enumerate")

**Config** — `CRISPRGuideEnumerationConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>pam</code> | <code>str</code> | <code>'NGG'</code> | IUPAC PAM motif without protospacer placeholder bases |
| <code>pam_side</code> | <code>Literal['three_prime', 'five_prime']</code> | <code>'three_prime'</code> | PAM position relative to the protospacer |
| <code>protospacer_length</code> | <code>int</code> | <code>20</code> | Number of PAM-free guide bases |
| <code>strands</code> | <code>Literal['both', 'plus', 'minus']</code> | <code>'both'</code> | Reference PAM-bearing strands to scan |
| <code>context_5p</code> | <code>int</code> | <code>4</code> | Exterior 5-prime context bases to return |
| <code>context_3p</code> | <code>int</code> | <code>3</code> | Exterior 3-prime context bases to return |

In [4]:
display_api_reference("crispr-guides-enumerate", "output", "run_crispr_guides_enumerate")

**Output** — `CRISPRGuideEnumerationOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[CRISPRGuideRegionResult]</code> | required | One result per input region |
| <code>pam</code> | <code>str</code> | required | Normalized IUPAC PAM grammar |
| <code>pam_side</code> | <code>Literal['three_prime', 'five_prime']</code> | required | PAM orientation |
| <code>protospacer_length</code> | <code>int</code> | required | PAM-free guide length |
| <code>strands</code> | <code>Literal['both', 'plus', 'minus']</code> | required | Scanned PAM strands |
| <code>context_5p</code> | <code>int</code> | required | Requested exterior 5-prime context |
| <code>context_3p</code> | <code>int</code> | required | Requested exterior 3-prime context |

**`CRISPRGuideRegionResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence_id</code> | <code>str</code> | required | Reference record or contig identifier |
| <code>region_start</code> | <code>int</code> | required | 1-indexed inclusive interval start |
| <code>region_end</code> | <code>int</code> | required | 1-indexed inclusive interval end |
| <code>reference_id</code> | <code>str &#124; None</code> | <code>None</code> | Assembly the coordinates refer to |
| <code>candidates</code> | <code>list[CRISPRGuideCandidate]</code> | <code>[]</code> | PAM-compatible protospacers |
| <code>ambiguous_windows_skipped</code> | <code>int</code> | <code>0</code> | Non-canonical windows excluded |

**`CRISPRGuideCandidate`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>protospacer</code> | <code>str</code> | required | PAM-free sequence in nuclease orientation |
| <code>pam</code> | <code>str</code> | required | Matched PAM sequence in nuclease orientation |
| <code>pam_side</code> | <code>Literal['three_prime', 'five_prime']</code> | required | PAM orientation |
| <code>pam_strand</code> | <code>Literal['+', '-']</code> | required | Reference strand carrying the PAM |
| <code>protospacer_start</code> | <code>int</code> | required | 1-indexed inclusive genomic start |
| <code>protospacer_end</code> | <code>int</code> | required | 1-indexed inclusive genomic end |
| <code>pam_start</code> | <code>int</code> | required | 1-indexed inclusive genomic start |
| <code>pam_end</code> | <code>int</code> | required | 1-indexed inclusive genomic end |
| <code>context_sequence</code> | <code>str</code> | required | Oriented target with exterior context |
| <code>context_start</code> | <code>int</code> | required | 1-indexed inclusive genomic context start |
| <code>context_end</code> | <code>int</code> | required | 1-indexed inclusive genomic context end |

## Enumerate SpCas9 Guides

Each region is supplied in reference plus-strand orientation with the 1-indexed coordinate of its first base. Every returned coordinate is derived from that, so plus- and minus-strand candidates are reported on the same reference axis.

In [5]:
# A synthetic interval with one 20-nt SpCas9 protospacer followed by TGG.
region = CRISPRGuideRegion(
    sequence_id="synthetic_interval",
    sequence="AAAAGACTGCATTGCATGACTGCATGGAAA",
    start=1,
    reference_id="synthetic-v1",
)
result = run_crispr_guides_enumerate(
    CRISPRGuideEnumerationInput(regions=region),
    CRISPRGuideEnumerationConfig(strands="plus"),
)
[(candidate.protospacer, candidate.pam, candidate.pam_strand) for candidate in result.results[0].candidates]

[('GACTGCATTGCATGACTGCA', 'TGG', '+')]

## Feeding an Activity Model

The default 4-nt and 3-nt exterior flanks make `context_sequence` the 30-nt window Rule Set 3 scores, with a 4-nt flank, 20-nt protospacer, 3-nt PAM, 3-nt flank. A candidate close to an interval edge keeps only the flank the interval provides, so check the length before passing it on.

In [6]:
candidate = result.results[0].candidates[0]
print(candidate.context_sequence, len(candidate.context_sequence))
print(f"protospacer {candidate.protospacer_start}-{candidate.protospacer_end} on {candidate.pam_strand}")

AAAAGACTGCATTGCATGACTGCATGGAAA 30
protospacer 5-24 on +


## Other Nucleases

The PAM is written in nuclease orientation as an IUPAC motif, so a five-prime PAM such as Cas12a `TTTV` only needs `pam_side="five_prime"`. Scanning `both` strands reports candidates from each reference strand against the same coordinates.

In [7]:
cas12a = run_crispr_guides_enumerate(
    CRISPRGuideEnumerationInput(
        regions=CRISPRGuideRegion(
            sequence_id="synthetic_cas12a",
            sequence="TTTAGACTGCATTGCATGACTGCA",
        )
    ),
    CRISPRGuideEnumerationConfig(
        pam="TTTV", pam_side="five_prime", strands="both", context_5p=0, context_3p=0
    ),
)
cas12a.results[0].candidates

[CRISPRGuideCandidate(protospacer='GACTGCATTGCATGACTGCA', pam='TTTA', pam_side='five_prime', pam_strand='+', protospacer_start=5, protospacer_end=24, pam_start=1, pam_end=4, context_sequence='TTTAGACTGCATTGCATGACTGCA', context_start=1, context_end=24)]

## Export Results

JSON keeps the per-region nesting. CSV flattens to one row per candidate, with its region's `sequence_id` in the first column.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="crispr_guide_candidates", export_path=output_dir, file_format="json")
print(f"Guide candidates exported to {output_dir}")

Guide candidates exported to example_output
